# Package

In [1]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# ----------------------------
# Nixtla MLForecast
# ----------------------------
from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals

# ----------------------------
# Model backend
# ----------------------------
from sklearn.linear_model import LinearRegression

# Importation des données

# Utilisation des données lags 12

In [2]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
from pathlib import Path
import pandas as pd
from feast import FeatureStore

def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())

# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(
        entity_df=entity_df,
        features=feature_refs,
        full_feature_names=True,
    ).to_df()

# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"

series_ids = [
    "BUSLOANS",
    "CPIAUCSL",
    "DPCERA3M086SBEA",
    "INDPRO",
    "M2SL",
    "OILPRICEX",
    "RPI",
    "SP500",
    "TB3MS",
    "UNRATE",
    "USREC",
]

# On veut les séries stationnarisées pour le modèle
FEATURE_REFS = ["stationary_value:value"]

# ----------------------------
# 1) Dates de référence (via UNRATE raw) pour avoir le vrai calendrier dispo
# ----------------------------
calendar = pd.date_range(start=START, end=END, freq=FREQ)

entity_df_unrate = pd.DataFrame({"series_id": ["UNRATE"] * len(calendar), "date": calendar})
df_unrate_dates = load_features_from_feast(entity_df_unrate, ["raw_value:value"])

dates = (
    pd.to_datetime(df_unrate_dates["date"], utc=True, errors="coerce")
      .dt.tz_convert(None)
      .dropna()
      .sort_values()
      .unique()
)

print("Nb dates:", len(dates))
print("Date min:", dates.min(), "| Date max:", dates.max())

# ----------------------------
# 2) Entity DF multi-séries × dates (long)
# ----------------------------
entity_df = (
    pd.MultiIndex.from_product([series_ids, dates], names=["series_id", "date"])
      .to_frame(index=False)
)

# ----------------------------
# 3) Fetch stationary features (long)
# ----------------------------
df_stationary = load_features_from_feast(entity_df, FEATURE_REFS)

df_stationary["date"] = (
    pd.to_datetime(df_stationary["date"], utc=True, errors="coerce")
      .dt.tz_convert(None)
)

value_col = "stationary_value__value"
if value_col not in df_stationary.columns:
    candidates = [c for c in df_stationary.columns if c.endswith("__value")]
    raise KeyError(f"Expected '{value_col}' not found. Candidates: {candidates}")

df_stationary = (
    df_stationary[["series_id", "date", value_col]]
    .rename(columns={value_col: "value"})
)

print("df_stationary shape:", df_stationary.shape)
print(df_stationary.head())

# ----------------------------
# 4) Dataset régression ciblé UNRATE
#    y = UNRATE (stationary)
#    exog = autres séries (contemporaines)
# ----------------------------
df_y = (
    df_stationary[df_stationary["series_id"] == "UNRATE"]
    .sort_values("date")
    .rename(columns={"value": "y"})
    .reset_index(drop=True)
)

df_x_long = df_stationary[df_stationary["series_id"] != "UNRATE"].copy()
df_x_long = df_x_long[df_x_long["date"].isin(df_y["date"])]

df_x = (
    df_x_long
    .pivot_table(index="date", columns="series_id", values="value", aggfunc="last")
    .reset_index()
)

df_model = (
    df_y[["date", "y"]]
    .merge(df_x, on="date", how="left")
    .dropna()
)

# ----------------------------
# 5) Format MLForecast (long + exog cols)
# ----------------------------
ts_lr = df_model.rename(columns={"date": "ds"})
ts_lr["unique_id"] = "UNRATE"

exog_cols = [c for c in ts_lr.columns if c not in ["unique_id", "ds", "y"]]
ts_lr = ts_lr[["unique_id", "ds", "y"] + exog_cols]

print("ts_lr shape:", ts_lr.shape)
print("Exog cols:", exog_cols)
ts_lr.head()

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
Nb dates: 801
Date min: 1959-01-01 00:00:00 | Date max: 2025-09-01 00:00:00
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df_stationary shape: (8679, 3)
  series_id       date     value
0  BUSLOANS 1960-01-01  0.011578
1    INDPRO 1960-01-01  0.091976
2     USREC 1960-01-01  0.000000
3      M2SL 1960-01-01  0.001323
4  CPIAUCSL 1960-01-01 -0.006156
ts_lr shape: (788, 13)
Exog cols: ['BUSLOANS', 'CPIAUCSL', 'DPCERA3M086SBEA', 'INDPRO', 'M2SL', 'OILPRICEX', 'RPI', 'SP500', 'TB3MS', 'USREC']


,unique_id,ds,y,BUSLOANS,CPIAUCSL,DPCERA3M086SBEA,INDPRO,M2SL,OILPRICEX,RPI,SP500,TB3MS,USREC
0,UNRATE,1960-01-01,-0.8,0.011578,-0.006156,0.001204,0.091976,0.001323,0.0,0.020977,0.017909,0.30,0.0
1,UNRATE,1960-02-01,-1.1,0.011905,-0.003767,0.006009,0.076960,0.002007,0.0,0.014565,-0.025663,-0.19,0.0
2,UNRATE,1960-03-01,-0.2,-0.008356,-0.005455,0.021240,0.007959,0.001324,0.0,0.006250,-0.070857,-1.18,0.0
3,UNRATE,1960-04-01,0.0,-0.009098,0.005090,0.033752,-0.025916,0.000634,0.0,0.006489,-0.040442,-1.12,0.0
4,UNRATE,1960-05-01,0.0,-0.000359,0.003383,0.009040,-0.018119,0.003977,0.0,0.007747,-0.010090,-0.67,1.0


In [3]:
ts_lr["ds"] = (
    pd.to_datetime(ts_lr["ds"])
      .dt.to_period("M")
      .dt.to_timestamp(how="start")
      .dt.normalize()
)

# Dictionnaire de modèle

In [30]:
from mlforecast import MLForecast
from sklearn.linear_model import Ridge

MLF_MODELS = {
    "RIDGE_EXOG_ONLY": lambda freq, *, alpha=1.0: MLForecast(
        models={"RIDGE": Ridge(alpha=float(alpha), random_state=0)},
        freq=freq,
        lags=[],               # EXOG-ONLY
        date_features=[],
    )
}

In [31]:
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from mlforecast.utils import PredictionIntervals


def _ensure_ms(x):
    x = pd.Timestamp(x)
    return x.to_period("M").to_timestamp(how="start").normalize()

def _n_windows_monthly(ds_start, ds_end):
    return (ds_end.year - ds_start.year) * 12 + (ds_end.month - ds_start.month) + 1

def _slice_cv_block(ts, cutoff_start, n_windows, h):
    """
    Construit une slice de données suffisante pour cross_validation sur un bloc :
    on garde toutes les obs jusqu'à cutoff_end + h (pour générer les ds prédites)
    """
    cutoff_end = cutoff_start + relativedelta(months=n_windows - 1)
    ds_end = cutoff_end + relativedelta(months=h)
    return ts[ts["ds"] <= ds_end].copy(), cutoff_end, ds_end

def _tune_alpha_on_train(ts_train, *, alpha_grid, cv=5):
    """
    Tuning alpha sur les données train (sklearn CV) — EXOG-ONLY.
    On entraîne sur toutes les lignes de ts_train (pas de fuite car <= cutoff_start du bloc).
    """
    # ts_train: columns = unique_id, ds, y, exog...
    # On enlève id/ds/y pour X
    drop_cols = {"unique_id", "ds", "y"}
    x_cols = [c for c in ts_train.columns if c not in drop_cols]
    X = ts_train[x_cols].values
    y = ts_train["y"].values

    grid = GridSearchCV(
        Ridge(fit_intercept=True, random_state=0),
        {"alpha": alpha_grid},
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1,
    )
    grid.fit(X, y)
    return float(grid.best_estimator_.alpha), float(-grid.best_score_)

def run_backtesting_h12_monthly_tune_every_36m(
    ts,
    *,
    freq,
    h=12,
    exp_start="1990-01-01",
    exp_end="2025-08-01",
    step_size=1,
    pi_windows=24,
    levels=[95],
    # tuning
    tune_every_months=36,
    alpha_mode="cv",                # "cv" ou float
    alpha_grid=np.logspace(-4, 4, 30),
    min_train_n=None,               # optionnel
):
    ts = ts.copy()

    # --- dates MS propres ---
    ts["ds"] = (
        pd.to_datetime(ts["ds"], errors="coerce")
          .dt.to_period("M")
          .dt.to_timestamp(how="start")
          .dt.normalize()
    )
    if ts["ds"].isna().any():
        bad = ts[ts["ds"].isna()].head()
        raise ValueError(f"Dates 'ds' invalides après parsing. Exemples:\n{bad}")

    exp_start = _ensure_ms(exp_start)
    exp_end   = _ensure_ms(exp_end)

    cutoff_start_all = exp_start - relativedelta(months=h)
    cutoff_end_all   = exp_end   - relativedelta(months=h)
    total_partitions = _n_windows_monthly(cutoff_start_all, cutoff_end_all)

    # anti-fuite : on coupe à exp_end
    ts = ts[ts["ds"] <= exp_end].copy()

    # Conformal PI
    pi = PredictionIntervals(h=h, n_windows=pi_windows, method="conformal_distribution")

    # --- découpage en blocs de 36 mois (ou moins à la fin) ---
    blocks = []
    remaining = total_partitions
    cur_cutoff_start = cutoff_start_all

    while remaining > 0:
        n_win = min(tune_every_months, remaining)
        blocks.append((cur_cutoff_start, n_win))
        cur_cutoff_start = cur_cutoff_start + relativedelta(months=n_win)
        remaining -= n_win

    all_bkts = []
    alpha_history = []   # par bloc
    cv_mae_history = []  # par bloc

    for block_idx, (cutoff_start_blk, n_windows_blk) in enumerate(blocks, start=1):
        # --- data slice utile pour ce bloc ---
        ts_blk, cutoff_end_blk, ds_end_blk = _slice_cv_block(ts, cutoff_start_blk, n_windows_blk, h)

        # --- train pour tuner (<= cutoff_start_blk) ---
        ts_train_for_tune = ts[ts["ds"] <= cutoff_start_blk].copy()
        if min_train_n is not None and len(ts_train_for_tune) < int(min_train_n):
            # pas assez d'historique : on skip
            continue

        # --- tuning alpha ---
        if alpha_mode == "cv":
            alpha_blk, cv_mae = _tune_alpha_on_train(ts_train_for_tune, alpha_grid=alpha_grid, cv=5)
        else:
            alpha_blk, cv_mae = float(alpha_mode), np.nan

        alpha_history.append(
            {"block": block_idx, "cutoff_start": cutoff_start_blk, "n_windows": n_windows_blk, "alpha": alpha_blk}
        )
        cv_mae_history.append(
            {"block": block_idx, "cutoff_start": cutoff_start_blk, "cv_mae": cv_mae}
        )

        # --- modèle Nixtla avec alpha figé sur le bloc ---
        mlf_blk = MLF_MODELS["RIDGE_EXOG_ONLY"](freq, alpha=alpha_blk)

        bkt_blk = mlf_blk.cross_validation(
            df=ts_blk,
            h=h,
            step_size=step_size,          # mensuel
            n_windows=n_windows_blk,      # seulement le bloc
            prediction_intervals=pi,
            level=levels,
            fitted=True,
            static_features=[],
            dropna=True,
        )

        # tag bloc + alpha utilisé
        bkt_blk["tune_block"] = block_idx
        bkt_blk["alpha_used"] = alpha_blk
        all_bkts.append(bkt_blk)

    if not all_bkts:
        return pd.DataFrame(), {"error": "Aucun bloc backtest produit (min_train_n trop grand ?)"}  # fallback

    bkt_df = pd.concat(all_bkts, ignore_index=True)

    # garder uniquement la plage d’expérience
    bkt_df = bkt_df[(bkt_df["ds"] >= exp_start) & (bkt_df["ds"] <= exp_end)].copy()
    bkt_df = bkt_df.sort_values(["unique_id", "ds", "cutoff"]).reset_index(drop=True)

    meta = {
        "h": h,
        "step_size": step_size,
        "exp_start": exp_start,
        "exp_end": exp_end,
        "cutoff_start": cutoff_start_all,
        "cutoff_end": cutoff_end_all,
        "partitions": total_partitions,
        "pi_windows": pi_windows,
        "tune_every_months": tune_every_months,
        "alpha_mode": alpha_mode,
        "alpha_grid": list(alpha_grid) if alpha_mode == "cv" else None,
        "alpha_history": alpha_history,
        "cv_mae_history": cv_mae_history,
    }

    return bkt_df, meta

# Run 

In [32]:
# ============================================================
# RUN – RIDGE (Nixtla MLForecast) | EXOG-ONLY
# FIT chaque mois (step=1), tuning alpha tous les 36 mois
# horizon 12, bornes exactes exp
# + ds unique (dernier cutoff)
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
print("PROJECT_ROOT =", PROJECT_ROOT.resolve())

H = 12
STEP_SIZE = 1
PI_WINDOWS = 3
LEVELS = [95]

EXP_START = "1990-01-01"
EXP_END   = "2025-08-01"

TUNE_EVERY_MONTHS = 36
ALPHA_MODE = "cv"  # ou 1.0
ALPHA_GRID = np.logspace(-4, 4, 30)

# ts_ridge: ton dataframe long Nixtla avec colonnes unique_id, ds, y + exog
ts_ridge = ts_lr.copy()  # si tu réutilises le même dataset que LR
ts_ridge["ds"] = (
    pd.to_datetime(ts_ridge["ds"], errors="coerce")
      .dt.to_period("M")
      .dt.to_timestamp(how="start")
      .dt.normalize()
)

# anti-fuite
ts_ridge = ts_ridge[ts_ridge["ds"] <= pd.Timestamp(EXP_END)].copy()

bkt_ridge, meta_ridge = run_backtesting_h12_monthly_tune_every_36m(
    ts=ts_ridge,
    freq=FREQ,
    h=H,
    exp_start=EXP_START,
    exp_end=EXP_END,
    step_size=STEP_SIZE,
    pi_windows=PI_WINDOWS,
    levels=LEVELS,
    tune_every_months=TUNE_EVERY_MONTHS,
    alpha_mode=ALPHA_MODE,
    alpha_grid=ALPHA_GRID,
    min_train_n=36,
)

print("✅ meta_ridge keys:", list(meta_ridge.keys()))
print("bkt_ridge rows:", len(bkt_ridge))

# ==========================
# ds unique: dernier cutoff
# ==========================
bkt_ridge_final = (
    bkt_ridge.sort_values(["unique_id", "ds", "cutoff"])
             .groupby(["unique_id", "ds"], as_index=False)
             .tail(1)
             .reset_index(drop=True)
)

print("bkt_ridge_final rows        :", len(bkt_ridge_final))
print("duplicates (unique_id, ds)  :", bkt_ridge_final.duplicated(["unique_id","ds"]).sum())
print("alphas used (unique)        :", sorted(bkt_ridge_final["alpha_used"].unique().tolist()))

bkt_ridge_final.head()

PROJECT_ROOT = D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook
✅ meta_ridge keys: ['h', 'step_size', 'exp_start', 'exp_end', 'cutoff_start', 'cutoff_end', 'partitions', 'pi_windows', 'tune_every_months', 'alpha_mode', 'alpha_grid', 'alpha_history', 'cv_mae_history']
bkt_ridge rows: 5070
bkt_ridge_final rows        : 428
duplicates (unique_id, ds)  : 0
alphas used (unique)        : [0.03039195382313198, 0.05736152510448681, 0.38566204211634725, 1.3738237958832638, 2.592943797404667, 4.893900918477489, 9.236708571873866]


,unique_id,ds,cutoff,y,RIDGE,RIDGE-lo-95,RIDGE-hi-95,tune_block,alpha_used
0,UNRATE,1990-01-01,1989-12-01,0.0,-0.374110,-0.607142,-0.141079,1,0.385662
1,UNRATE,1990-02-01,1990-01-01,0.1,-0.415840,-1.277387,0.445707,1,0.385662
2,UNRATE,1990-03-01,1990-02-01,0.2,-0.424608,-1.467775,0.618558,1,0.385662
3,UNRATE,1990-04-01,1990-03-01,0.2,-0.334295,-1.268968,0.600377,1,0.385662
4,UNRATE,1990-05-01,1990-04-01,0.2,-0.069912,-0.794322,0.654498,1,0.385662


# Sauvegarde

# Graphique

In [34]:
import pandas as pd
from utilsforecast.plotting import plot_series

# ============================================================
# Prepare obs (RIDGE)
# ============================================================
df_plot = df_ridge_forecasts.copy()
df_plot["date"] = pd.to_datetime(df_plot["date"], errors="coerce")

if df_plot["date"].isna().any():
    bad = df_plot[df_plot["date"].isna()].head()
    raise ValueError(f"Dates invalides dans df_ridge_forecasts['date']:\n{bad}")

# sécurité: pas de doublons
df_plot = df_plot.sort_values(["series_id", "date"]).reset_index(drop=True)
if df_plot.duplicated(["series_id", "date"]).any():
    # si jamais, garder la dernière ligne (cutoff le plus récent) si colonne existe
    if "cutoff" in df_plot.columns:
        df_plot = (df_plot.sort_values(["series_id","date","cutoff"])
                         .drop_duplicates(["series_id","date"], keep="last")
                         .reset_index(drop=True))
    else:
        df_plot = df_plot.drop_duplicates(["series_id","date"], keep="last").reset_index(drop=True)

df_obs = (
    df_plot.rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_obs": "y",
    })[["unique_id", "ds", "y"]]
)

# ============================================================
# Prepare forecast + PI (RIDGE)
# ============================================================
df_fcst = (
    df_plot.rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_hat_ridge": "RIDGE",
        "y_hat_ridge_lo_95": "RIDGE-lo-95",
        "y_hat_ridge_hi_95": "RIDGE-hi-95",
    })[["unique_id", "ds", "RIDGE", "RIDGE-lo-95", "RIDGE-hi-95"]]
)

# ============================================================
# Plot
# ============================================================
fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(height=400)

for trace in fig.data:
    if trace.name == "y":
        trace.name = "Unemployment rate (stationary)"
    elif trace.name == "RIDGE":
        trace.name = f"Ridge Regression (exog only, alpha={RIDGE_ALPHA})" if "RIDGE_ALPHA" in globals() else "Ridge Regression (exog only)"
    elif "level_95" in (trace.name or "").lower():
        trace.name = "95% Prediction Interval"

fig.show()

# Evaluaer 

In [35]:
import numpy as np
import pandas as pd

# -----------------------------
# 1) One forecast per target date ds
#    (keep the most recent cutoff for each ds)
# -----------------------------
bkt_ridge_final = (
    bkt_ridge_final
    .sort_values(["unique_id", "ds", "cutoff"])
    .groupby(["unique_id", "ds"], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

# -----------------------------
# 2) Segments + ALL
# -----------------------------
segments = [
    ("1990-01-01", "1999-12-31", "1990-1999"),
    ("2000-01-01", "2008-07-31", "2000-2008"),
    ("2008-08-01", "2019-12-31", "2008-2019"),
    ("2020-01-01", None,         "2020-fin"),
]

def mae_safe(y_true, y_pred):
    df = pd.DataFrame({"y": y_true, "yhat": y_pred}).dropna()
    if len(df) == 0:
        return np.nan
    return float(np.mean(np.abs(df["y"].to_numpy() - df["yhat"].to_numpy())))

rows = []

# --- ALL ---
rows.append({
    "period": "ALL",
    "n_obs": int(len(bkt_ridge_final)),
    "MAE_RIDGE": mae_safe(bkt_ridge_final["y"], bkt_ridge_final["RIDGE"]),
})

# --- Segments ---
for start, end, label in segments:
    mask = bkt_ridge_final["ds"] >= pd.Timestamp(start)
    if end is not None:
        mask &= bkt_ridge_final["ds"] <= pd.Timestamp(end)

    df_seg = bkt_ridge_final.loc[mask]

    rows.append({
        "period": label,
        "n_obs": int(len(df_seg)),
        "MAE_RIDGE": mae_safe(df_seg["y"], df_seg["RIDGE"]),
    })

df_mae_ridge = pd.DataFrame(rows)

# Ordre propre
order = ["ALL"] + [s[2] for s in segments]
df_mae_ridge["period"] = pd.Categorical(df_mae_ridge["period"], categories=order, ordered=True)
df_mae_ridge = df_mae_ridge.sort_values("period").reset_index(drop=True)

df_mae_ridge

,period,n_obs,MAE_RIDGE
0,ALL,428,0.798519
1,1990-1999,120,0.498683
2,2000-2008,103,0.423158
3,2008-2019,137,0.803542
4,2020-fin,68,1.886085


Très bien partie

# Sauvegarde

In [15]:
# ============================================================
# (ADD) 6) Save artifacts & outputs (Linear Regression)
# ============================================================

from datetime import datetime
import json

# ----------------------------
# Model identification
# ----------------------------
SERIES_ID = "UNRATE"
MODEL_TAG = "lr_lag12_exog"   # 🔑 clair et extensible

# ----------------------------
# Directories
# ----------------------------
OUTPUT_FORECASTS_DIR = PROJECT_ROOT / "outputs" / "forecasts"

ARTIFACT_CONFIGS_DIR = PROJECT_ROOT / "artifacts" / "configs"
ARTIFACT_CV_DIR      = PROJECT_ROOT / "artifacts" / "cv"
ARTIFACT_META_DIR    = PROJECT_ROOT / "artifacts" / "metadata"

OUTPUT_FORECASTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_CV_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_META_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 1) Outputs (OOS forecasts)
# ----------------------------
oos_path = OUTPUT_FORECASTS_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_oos_forecasts.parquet"
df_lr_forecasts.to_parquet(oos_path, index=False)

# ----------------------------
# 2) Artifacts – raw backtesting output
# ----------------------------
bkt_path = ARTIFACT_CV_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_bkt_raw.parquet"
bkt_lr.to_parquet(bkt_path, index=False)

# ----------------------------
# 3) Run configuration (reproducibility)
# ----------------------------
run_config = {
    "model": "LinearRegression",
    "framework": "Nixtla-MLForecast",
    "target": SERIES_ID,
    "stationary": True,
    "lags": [12],
    "exogenous_variables": [
        c for c in ts_lr.columns if c not in ["unique_id", "ds", "y"]
    ],
    "horizon": H,
    "step_size": STEP_SIZE,
    "partitions": PARTITIONS,
    "prediction_intervals": {
        "method": "conformal_distribution",
        "levels": LEVELS,
        "n_windows": PI_WINDOWS,
    },
    "frequency": FREQ,
}

cfg_path = ARTIFACT_CONFIGS_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_config.json"
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

# ----------------------------
# 4) Metadata – run info
# ----------------------------
meta_path = ARTIFACT_META_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_run_info.json"
run_info = {
    "run_utc": datetime.utcnow().isoformat(),
    "project_root": str(PROJECT_ROOT.resolve()),
    "model_tag": MODEL_TAG,
    "files": {
        "oos_forecasts": str(oos_path.resolve()),
        "cv_raw": str(bkt_path.resolve()),
        "config": str(cfg_path.resolve()),
    },
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2)

print("✅ Saved:")
print(" - OOS forecasts :", oos_path.name)
print(" - CV raw        :", bkt_path.name)
print(" - Config        :", cfg_path.name)
print(" - Metadata      :", meta_path.name)


✅ Saved:
 - OOS forecasts : unrate_lr_lag12_exog_oos_forecasts.parquet
 - CV raw        : unrate_lr_lag12_exog_bkt_raw.parquet
 - Config        : unrate_lr_lag12_exog_config.json
 - Metadata      : unrate_lr_lag12_exog_run_info.json


C:\Users\Mita\AppData\Local\Temp\ipykernel_6056\677556810.py:72: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



# Graphique

## Résultat
On voit notre modèle AR1 est très basique, il suit juste la direction du taux de chômage. Avec ce baseline, on s'aperçoit des informations très importantes pour orienter notre expérimentation. 

De 1990 à 2007, l'économie américaine a été stable. En 2008, elle a été frappée par la crise de Subprime. En 2019, çà été la crise de Coronavirus. Qu'est-ce qu'on peut dire de ces trois périodes? 

Globalement, le modèle auto-régressif reste proche des observations en période de stabilité. C'est une bonne capacité à capter la dynamique du chômage.

Lors des ruptures structurelles des deux crises, la qualité des prévisions se dégrade et les intervalles de prédiction s’élargissent. Ce qui traduit une incertitude de plus en plus élevée.

Le modèle capte la direction des variations, mais sa fiabilité diminue en période de crise, sans masquer cette incertitude.

Cette étude constitue un **sanity check du système de prévision**. Elle valide le comportement attendu du modèle et la cohérence du pipeline.

Une approche plus complexe est attendu. 

## Next
Essayons d'optimiser le paramètre "p" de AR pour confirmer notre analyse. 